# juristische Regeln mit spaCy

in-context learning (ICL)


"Mieter + verpflichtet + zahlen"

In [4]:
import spacy

nlp = spacy.load("de_core_news_sm")

text = "Der Mieter ist verpflichtet, die monatliche Miete bis zum dritten Werktag zu zahlen."

doc = nlp(text)

for token in doc:
    print(token.text, token.lemma_, token.pos_)

if "verpflichten" in [t.lemma_ for t in doc]:
    print("⚖️ mögliche Pflichtnorm erkannt")

Der der DET
Mieter Mieter NOUN
ist sein AUX
verpflichtet verpflichten VERB
, -- PUNCT
die der DET
monatliche monatlich ADJ
Miete Miete NOUN
bis bis ADP
zum zu ADP
dritten dritter ADJ
Werktag Werktag NOUN
zu zu PART
zahlen zahlen VERB
. -- PUNCT
⚖️ mögliche Pflichtnorm erkannt


# nlp-Funktion erwartet einen einzelnen String (Text) 

In [8]:
text = """
Der Mietvertrag beginnt am 01.01.2025.
Der Mieter ist verpflichtet die Miete zu zahlen.
Die Kündigungsfrist beträgt drei Monate.
"""
# .strip() entfernt Leerzeilen am Anfang/Ende, .replace() ersetzt Umbrüche durch Leerzeichen
clean_text = text.strip().replace("\n", " ")
doc = nlp(clean_text)

#doc = nlp(text)

sentences = [sent.text for sent in doc.sents]

print(sentences)

['Der Mietvertrag beginnt am 01.01.2025.', 'Der Mieter ist verpflichtet die Miete zu zahlen.', 'Die Kündigungsfrist beträgt drei Monate.']


# text1=" ".join(text)

In [12]:
text = [
"Der Mietvertrag beginnt am 01.01.2025.",
"Der Mieter ist verpflichtet die Miete zu zahlen.",
"Die Kündigungsfrist beträgt drei Monate."
]
text1=" ".join(text)
print(text1)
doc = nlp(text1)

sentences = [sent.text for sent in doc.sents]

print(sentences)

Der Mietvertrag beginnt am 01.01.2025. Der Mieter ist verpflichtet die Miete zu zahlen. Die Kündigungsfrist beträgt drei Monate.
['Der Mietvertrag beginnt am 01.01.2025.', 'Der Mieter ist verpflichtet die Miete zu zahlen.', 'Die Kündigungsfrist beträgt drei Monate.']


# Hugging Face Modell für juristische Klassifikation

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "bert-base-german-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

labels = [
    "Kündigung",
    "Zahlungspflicht",
    "Haftung",
    "Vertragsbeginn"
]

text = "Der Mieter ist verpflichtet, die monatliche Miete bis zum dritten Werktag zu zahlen."

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
pred = torch.argmax(logits).item()

print("Vorhersage:", labels[pred])

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\wug2si\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wug2si\.cache\huggingface\hub\models--bert-base-german-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Vorhersage: Haftung


In [17]:
import de_core_news_sm
print(de_core_news_sm.__file__)


C:\0_DA\Python312\Lib\site-packages\de_core_news_sm\__init__.py


In [20]:
#!python -m spacy download de_core_news_md

In [27]:
import spacy
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

############################################
# 1. spaCy Modell laden
############################################

nlp = spacy.load("de_core_news_md")

############################################
# 2. Trainingsdaten (50 juristische Sätze)
############################################

data = [
("Der Mieter muss die Miete bis zum dritten Werktag zahlen.", "Zahlungspflicht"),
("Die monatliche Zahlung der Miete ist verpflichtend.", "Zahlungspflicht"),
("Der Käufer ist verpflichtet den Kaufpreis zu entrichten.", "Zahlungspflicht"),
("Die Rechnung ist innerhalb von 14 Tagen zu bezahlen.", "Zahlungspflicht"),
("Der Kunde muss die Gebühr überweisen.", "Zahlungspflicht"),
("Der Schuldner hat die Summe fristgerecht zu zahlen.", "Zahlungspflicht"),
("Die Zahlung erfolgt monatlich.", "Zahlungspflicht"),
("Der Betrag ist sofort fällig.", "Zahlungspflicht"),
("Die Zahlungspflicht entsteht mit Vertragsschluss.", "Zahlungspflicht"),
("Die Miete ist monatlich zu überweisen.", "Zahlungspflicht"),
("Die Zahlung muss spätestens am Monatsanfang erfolgen.", "Zahlungspflicht"),
("Der Nutzer verpflichtet sich zur Zahlung der Gebühr.", "Zahlungspflicht"),
("Der Käufer zahlt den Preis bei Lieferung.", "Zahlungspflicht"),

("Der Vertrag kann mit einer Frist von drei Monaten gekündigt werden.", "Kündigung"),
("Der Mieter darf das Mietverhältnis kündigen.", "Kündigung"),
("Die Kündigung muss schriftlich erfolgen.", "Kündigung"),
("Der Vertrag endet durch Kündigung.", "Kündigung"),
("Beide Parteien können den Vertrag kündigen.", "Kündigung"),
("Die Kündigungsfrist beträgt vier Wochen.", "Kündigung"),
("Der Arbeitgeber kündigt das Arbeitsverhältnis.", "Kündigung"),
("Der Kunde kann den Vertrag jederzeit kündigen.", "Kündigung"),
("Die Kündigung erfolgt zum Monatsende.", "Kündigung"),
("Eine außerordentliche Kündigung ist möglich.", "Kündigung"),
("Der Vertrag wird durch Kündigung beendet.", "Kündigung"),
("Der Nutzer hat ein Kündigungsrecht.", "Kündigung"),
("Die Kündigung muss fristgerecht erfolgen.", "Kündigung"),

("Der Verkäufer haftet für Sachmängel.", "Haftung"),
("Die Haftung ist auf Vorsatz beschränkt.", "Haftung"),
("Der Anbieter übernimmt keine Haftung.", "Haftung"),
("Die Haftung für Schäden ist ausgeschlossen.", "Haftung"),
("Der Hersteller haftet für Fehler.", "Haftung"),
("Die Haftung ist gesetzlich geregelt.", "Haftung"),
("Das Unternehmen haftet für Schäden.", "Haftung"),
("Die Partei haftet für Vertragsverletzungen.", "Haftung"),
("Die Haftung umfasst auch Folgeschäden.", "Haftung"),
("Der Betreiber haftet für Datenverlust.", "Haftung"),
("Die Haftung ist begrenzt.", "Haftung"),
("Der Anbieter haftet nur bei grober Fahrlässigkeit.", "Haftung"),

("Der Vertrag beginnt am 1. Januar.", "Vertragsbeginn"),
("Das Mietverhältnis startet am 01.01.2025.", "Vertragsbeginn"),
("Der Vertrag tritt sofort in Kraft.", "Vertragsbeginn"),
("Der Beginn des Vertrags ist der 1. März.", "Vertragsbeginn"),
("Das Abonnement startet heute.", "Vertragsbeginn"),
("Der Vertrag gilt ab Unterzeichnung.", "Vertragsbeginn"),
("Die Laufzeit beginnt am Tag der Registrierung.", "Vertragsbeginn"),
("Der Mietvertrag startet nächste Woche.", "Vertragsbeginn"),
("Die Vereinbarung beginnt mit der Zahlung.", "Vertragsbeginn"),
("Der Vertrag wird ab morgen wirksam.", "Vertragsbeginn"),
("Die Nutzung beginnt nach Aktivierung.", "Vertragsbeginn"),
("Der Vertrag startet mit Vertragsabschluss.", "Vertragsbeginn")
]

############################################
# 3. Label Mapping
############################################

labels = ["Zahlungspflicht","Kündigung","Haftung","Vertragsbeginn"]
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

############################################
# 4. spaCy Preprocessing
############################################

# lemmasierung ist nicht nötig für Transformer, hier nur als Demo:

texts = []
y = []

for text,label in data:
    doc = nlp(text)
    cleaned = " ".join([t.lemma_ for t in doc  if not t.is_punct and not t.is_space])
    #print("cleaned:", cleaned)
    texts.append(cleaned)
    y.append(label2id[label])
    
for i, t in enumerate(texts):   
    if i < 4: print("texts cleaned:", t)
    
############################################
# 5. HuggingFace Tokenizer
############################################

model_name = "bert-base-german-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

encodings = tokenizer(texts, truncation=True, padding=True)

############################################
# 6. Dataset Klasse
############################################

class LegalDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

dataset = LegalDataset(encodings,y)

loader = DataLoader(dataset,batch_size=8,shuffle=True)

############################################
# 7. Modell laden
############################################

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels)
)

############################################
# 8. Training
############################################

optimizer = AdamW(model.parameters(),lr=2e-5)

model.train()
epoches=10
for epoch in range(epoches):

    total_loss = 0

    for batch in loader:

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss)

############################################
# 9. Prediction
############################################

model.eval()

test_text = "Der Mieter muss die monatliche Miete bezahlen."

doc = nlp(test_text)
cleaned = " ".join([t.lemma_ for t in doc])

inputs = tokenizer(cleaned, return_tensors="pt")

with torch.no_grad():

    outputs = model(**inputs)

logits = outputs.logits
pred = torch.argmax(logits).item()

print("\nText:",test_text)
print("Vorhersage:",id2label[pred])

texts cleaned: der Mieter mussen der Miete bis zu dritter Werktag zahlen
texts cleaned: der monatlich Zahlung der Miete sein verpflichtend
texts cleaned: der Käufer sein verpflichten der Kaufpreis zu entrichten
texts cleaned: der Rechnung sein innerhalb von 14 Tag zu bezahlen


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Loss: 9.602489233016968
Epoch 2 Loss: 6.350900948047638
Epoch 3 Loss: 4.151809334754944
Epoch 4 Loss: 2.4645614624023438
Epoch 5 Loss: 1.3181490153074265
Epoch 6 Loss: 0.7536281496286392
Epoch 7 Loss: 0.44766515120863914
Epoch 8 Loss: 0.30875295773148537
Epoch 9 Loss: 0.23928873427212238
Epoch 10 Loss: 0.18174392729997635

Text: Der Mieter muss die monatliche Miete bezahlen.
Vorhersage: Zahlungspflicht


# Cuda nutzung

In [30]:

import spacy
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

# 0. Device Setup (Prüfen ob GPU verfügbar)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Gerät: {device}")

############################################
# 1. spaCy Modell laden
############################################

nlp = spacy.load("de_core_news_md")

############################################
# 2. Trainingsdaten (50 juristische Sätze)
############################################

data = [
("Der Mieter muss die Miete bis zum dritten Werktag zahlen.", "Zahlungspflicht"),
("Die monatliche Zahlung der Miete ist verpflichtend.", "Zahlungspflicht"),
("Der Käufer ist verpflichtet den Kaufpreis zu entrichten.", "Zahlungspflicht"),
("Die Rechnung ist innerhalb von 14 Tagen zu bezahlen.", "Zahlungspflicht"),
("Der Kunde muss die Gebühr überweisen.", "Zahlungspflicht"),
("Der Schuldner hat die Summe fristgerecht zu zahlen.", "Zahlungspflicht"),
("Die Zahlung erfolgt monatlich.", "Zahlungspflicht"),
("Der Betrag ist sofort fällig.", "Zahlungspflicht"),
("Die Zahlungspflicht entsteht mit Vertragsschluss.", "Zahlungspflicht"),
("Die Miete ist monatlich zu überweisen.", "Zahlungspflicht"),
("Die Zahlung muss spätestens am Monatsanfang erfolgen.", "Zahlungspflicht"),
("Der Nutzer verpflichtet sich zur Zahlung der Gebühr.", "Zahlungspflicht"),
("Der Käufer zahlt den Preis bei Lieferung.", "Zahlungspflicht"),

("Der Vertrag kann mit einer Frist von drei Monaten gekündigt werden.", "Kündigung"),
("Der Mieter darf das Mietverhältnis kündigen.", "Kündigung"),
("Die Kündigung muss schriftlich erfolgen.", "Kündigung"),
("Der Vertrag endet durch Kündigung.", "Kündigung"),
("Beide Parteien können den Vertrag kündigen.", "Kündigung"),
("Die Kündigungsfrist beträgt vier Wochen.", "Kündigung"),
("Der Arbeitgeber kündigt das Arbeitsverhältnis.", "Kündigung"),
("Der Kunde kann den Vertrag jederzeit kündigen.", "Kündigung"),
("Die Kündigung erfolgt zum Monatsende.", "Kündigung"),
("Eine außerordentliche Kündigung ist möglich.", "Kündigung"),
("Der Vertrag wird durch Kündigung beendet.", "Kündigung"),
("Der Nutzer hat ein Kündigungsrecht.", "Kündigung"),
("Die Kündigung muss fristgerecht erfolgen.", "Kündigung"),

("Der Verkäufer haftet für Sachmängel.", "Haftung"),
("Die Haftung ist auf Vorsatz beschränkt.", "Haftung"),
("Der Anbieter übernimmt keine Haftung.", "Haftung"),
("Die Haftung für Schäden ist ausgeschlossen.", "Haftung"),
("Der Hersteller haftet für Fehler.", "Haftung"),
("Die Haftung ist gesetzlich geregelt.", "Haftung"),
("Das Unternehmen haftet für Schäden.", "Haftung"),
("Die Partei haftet für Vertragsverletzungen.", "Haftung"),
("Die Haftung umfasst auch Folgeschäden.", "Haftung"),
("Der Betreiber haftet für Datenverlust.", "Haftung"),
("Die Haftung ist begrenzt.", "Haftung"),
("Der Anbieter haftet nur bei grober Fahrlässigkeit.", "Haftung"),

("Der Vertrag beginnt am 1. Januar.", "Vertragsbeginn"),
("Das Mietverhältnis startet am 01.01.2025.", "Vertragsbeginn"),
("Der Vertrag tritt sofort in Kraft.", "Vertragsbeginn"),
("Der Beginn des Vertrags ist der 1. März.", "Vertragsbeginn"),
("Das Abonnement startet heute.", "Vertragsbeginn"),
("Der Vertrag gilt ab Unterzeichnung.", "Vertragsbeginn"),
("Die Laufzeit beginnt am Tag der Registrierung.", "Vertragsbeginn"),
("Der Mietvertrag startet nächste Woche.", "Vertragsbeginn"),
("Die Vereinbarung beginnt mit der Zahlung.", "Vertragsbeginn"),
("Der Vertrag wird ab morgen wirksam.", "Vertragsbeginn"),
("Die Nutzung beginnt nach Aktivierung.", "Vertragsbeginn"),
("Der Vertrag startet mit Vertragsabschluss.", "Vertragsbeginn")
]

############################################
# 3. Label Mapping
############################################

labels = ["Zahlungspflicht","Kündigung","Haftung","Vertragsbeginn"]
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

############################################
# 4. spaCy Preprocessing
############################################

# lemmasierung ist nicht nötig für Transformer, hier nur als Demo:

texts = []
y = []

for text,label in data:
    doc = nlp(text)
    cleaned = " ".join([t.lemma_ for t in doc  if not t.is_punct and not t.is_space])
    #print("cleaned:", cleaned)
    texts.append(cleaned)
    y.append(label2id[label])
    
for i, t in enumerate(texts):   
    if i < 4: print("texts cleaned:", t)
    
############################################
# 5. HuggingFace Tokenizer
############################################

model_name = "bert-base-german-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

encodings = tokenizer(texts, truncation=True, padding=True)

############################################
# 6. Dataset Klasse
############################################

class LegalDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

dataset = LegalDataset(encodings,y)

loader = DataLoader(dataset,batch_size=8,shuffle=True)

############################################
# 7. Modell laden
############################################

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels)
)
model.to(device)  # <--- WICHTIG: Modell auf GPU

############################################
# 8. Training
############################################

optimizer = AdamW(model.parameters(),lr=2e-5)

model.train()
epoches=10
for epoch in range(epoches):

    total_loss = 0

    for batch in loader:
        # Alle Batch-Daten (Input IDs, Attention Mask, Labels) auf die GPU schieben
        batch = {k: v.to(device) for k, v in batch.items()} # <--- WICHTIG

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f}")

############################################
# 9. Prediction
############################################

model.eval()

test_text = "Der Mieter muss die monatliche Miete bezahlen."

doc = nlp(test_text)
cleaned = " ".join([t.lemma_ for t in doc])

# Input auf GPU schieben
inputs = tokenizer(cleaned, return_tensors="pt").to(device) # <--- WICHTIG

with torch.no_grad():

    outputs = model(**inputs)

logits = outputs.logits
pred = torch.argmax(logits).item()

print("\n--- Ergebnis ---")
print(f"Text: {test_text}")
print(f"Vorhersage: {id2label[pred]}")


Nutze Gerät: cuda
texts cleaned: der Mieter mussen der Miete bis zu dritter Werktag zahlen
texts cleaned: der monatlich Zahlung der Miete sein verpflichtend
texts cleaned: der Käufer sein verpflichten der Kaufpreis zu entrichten
texts cleaned: der Rechnung sein innerhalb von 14 Tag zu bezahlen


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-german-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 - Loss: 9.4293
Epoch 2/10 - Loss: 6.7224
Epoch 3/10 - Loss: 4.2395
Epoch 4/10 - Loss: 2.5483
Epoch 5/10 - Loss: 1.2678
Epoch 6/10 - Loss: 0.6838
Epoch 7/10 - Loss: 0.4270
Epoch 8/10 - Loss: 0.3220
Epoch 9/10 - Loss: 0.2386
Epoch 10/10 - Loss: 0.1991

--- Ergebnis ---
Text: Der Mieter muss die monatliche Miete bezahlen.
Vorhersage: Zahlungspflicht
